# NB_lab_test_trends
**Gold Table:** `caresync_databricks.gold.lab_test_trends`  
**Grain:** One row per test type (`test_name`) per calendar month  
**Driving Table:** `caresync_databricks.silver.lab_results` (grouped by `test_name`, year/month of `test_date`)  
**Load Strategy:** Full rebuild — `CREATE OR REPLACE TABLE ... AS SELECT`  
**Joins:**
- Optional LEFT JOIN `calendar` on `date_key = date_format(first-of-month, 'yyyyMMdd')`

**Window Function:** `LAG()` partitioned by `test_name`, ordered by `year`, `month` — month-over-month trend  
**Source:** STTM — Silver to Gold (CareSync Health Network) v1.0

In [0]:
dbutils.widgets.text("catalog", "caresync_databricks", "Catalog Name")
catalog = dbutils.widgets.get("catalog")
print(f"Using catalog: {catalog}")

In [0]:
%sql
CREATE OR REPLACE TABLE caresync_databricks.gold.lab_test_trends AS

WITH monthly_agg AS (
    SELECT
        lr.test_name,
        YEAR(lr.test_date)                                                  AS year,
        MONTH(lr.test_date)                                                 AS month,
        COUNT(*)                                                            AS total_tests,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)          AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)          AS critical_test_count
    FROM caresync_databricks.silver.lab_results lr
    GROUP BY lr.test_name, YEAR(lr.test_date), MONTH(lr.test_date)
),

with_rates AS (
    SELECT
        ma.test_name,
        ma.year,
        ma.month,
        c.month_name,
        CONCAT('Q', c.quarter)                                              AS quarter_name,
        ma.total_tests,
        ma.abnormal_test_count,
        ma.critical_test_count,
        CAST(ma.abnormal_test_count * 100.0 / ma.total_tests AS DECIMAL(5,2))   AS abnormal_rate_pct,
        CAST(ma.critical_test_count * 100.0 / ma.total_tests AS DECIMAL(5,2))   AS critical_rate_pct
    FROM monthly_agg ma
    LEFT JOIN caresync_databricks.silver.calendar c
        ON c.date_key = CAST(DATE_FORMAT(MAKE_DATE(ma.year, ma.month, 1), 'yyyyMMdd') AS INT)
)

SELECT
    test_name,
    year,
    month,
    month_name,
    quarter_name,
    total_tests,
    abnormal_test_count,
    critical_test_count,
    abnormal_rate_pct,
    critical_rate_pct,
    CAST(
        LAG(abnormal_rate_pct) OVER (
            PARTITION BY test_name ORDER BY year, month
        ) AS DECIMAL(5,2)
    )                                                                       AS prev_month_abnormal_rate_pct,
    CAST(
        abnormal_rate_pct - LAG(abnormal_rate_pct) OVER (
            PARTITION BY test_name ORDER BY year, month
        ) AS DECIMAL(6,2)
    )                                                                       AS abnormal_rate_change_pct
FROM with_rates

In [0]:
print("[STEP 4] Computing processed record count")
record_count = spark.table(f"caresync_databricks.gold.lab_test_trends").count()
print(f"[STEP 4 COMPLETE] Processed record count: {record_count}")
dbutils.notebook.exit(str(record_count))